In [149]:
print("📦 Installing required packages...")

!pip install --upgrade pip
!pip install -q openai

# Note: You may see a warning about requests version - this is safe to ignore
print("✅ Installation complete!\n")

📦 Installing required packages...
✅ Installation complete!



<h2 style="text-align: center;">
  <span style="display: inline-block; border-bottom: 2px solid black; padding-bottom: 5px;">
    Task 1
  </span>
</h2>

### **My Rubric:**

    - Criterion

        - Fluency
            - Coherent text, which sounds natural, clear, and easy to read: Good
            - Still as above, but slighly less fluent and clear:            OK
            - Gliches of uncoherent text, or awkward language:              Bad

        - Grammar
            - Grammatically correct, with punctuations, and uppercase where needed:                 Good
            - Still basically correct, but with minor glich or two (e.g. none perfect punctuation): OK
            - More than two grammaticall gliches:                                                   Bad

        - Tone
            - Matches friendly, credible sales voice, including poilte words and subtle enthusiasm:        Good
            - Still nice and decent, but lacking some of the aspects mentioned above:                      OK
            - Miss too many positive aspects, described above, or contains some less friendly expressions: Bad

        - Length
            - 50-90:                              Good
            - 40-49 or 91-100:                    OK
            - shorter than 40 or longer than 100: Bad

        - Grounding
            - No hallucinations, the description can be directly inferred from the product name or features:     Good
            - As above, but with possible minor 'creativity' (e.g. use synonyms, 'metal' instead of 'aluminum'): OK
            - Contains hallucinations, or description that can be hardly inferred from the content:              Bad

        - Latency:
            - shorter than 1 second:    Good
            - between 1 to 1.5 seconds: OK
            - longer than 1.5 seconds:  Bad

        - Cost
            - less than $0.00003:            Good
        - between $0.00003 to $0.000035: OK
        - more than $0.000035:           Bad

    - Pass / Fail Formula:
        - Fail
            - One of the criterions is rated 'Bad'
        - Pass
            - at least 3 criterions are rated 'Good', and there is no 'Bad' rate for any criterion


<h2 style="text-align: center;">
  <span style="display: inline-block; border-bottom: 2px solid black; padding-bottom: 5px;">
    Task 2
  </span>
</h2>

Define the prompts and the code, which generate description for the products table

In [151]:
# Read the products table and perform some pre-processing.
import pandas as pd
import unicodedata
import json

TABLE_PATH = "/home/yaron/WorkEnv/data/nebius/tasks/task_1_product_dataset.csv"
DESCRIPTIONS_TABLE = "/home/yaron/WorkEnv/data/nebius/tasks/assignment_01"
NAME_FIELD = "Name"
ATTR_FIELD = "attributes"
GEN_DESC_FIELD = "generated_description"
MATERIAL_FIELD = "material"
WARRAN_FIELD = "warranty"
TO_BE_REP = "Product_attribute_list"

# Normalize the text for removing Unicode quirks, as well as better presentation of the columns.
def normalize_text(s):
    if isinstance(s, str):
        s = unicodedata.normalize("NFKC", s)
        return s.replace("\u202f", " ").replace("\u2010", " ").replace("\xa0", " ")
    return s

# Arrange columns presentation and normalize all texts.
def preprocess_table(df: pd.DataFrame) -> pd.DataFrame:
    df[NAME_FIELD] = df.index
    df.index = [i for i in range(len(df))]
    df = df.rename(columns={TO_BE_REP: ATTR_FIELD})
    df = df[[NAME_FIELD, ATTR_FIELD, MATERIAL_FIELD, WARRAN_FIELD]].copy()
    for field in [NAME_FIELD, ATTR_FIELD, MATERIAL_FIELD, WARRAN_FIELD]:
        df[field] = df[field].apply(lambda x: normalize_text(x))
    return df

# Read the table.
products_df = pd.read_csv(TABLE_PATH, index_col=0)
print("Original Table:")
products_df.head(5)

Original Table:


,Product_attribute_list,material,warranty
product_name,,,
Apple iPhone 15 Pro,"features: A17 Pro chip, 120 Hz ProMotion displ...","titanium frame, Ceramic Shield glass",1‑year limited warranty
Samsung Galaxy S24 Ultra,"features: 200 MP camera, S‑Pen support, 120 Hz...","Armor Aluminum frame, Gorilla Glass Victus",1‑year limited warranty
Google Pixel 8 Pro,"features: Tensor G3 chip, Magic Eraser, 50 MP ...","matte glass back, aluminum frame",1‑year limited warranty
Sony WH‑1000XM5 Headphones,"features: active noise cancelling, 30 hr batte...",synthetic leather earcups,1‑year limited warranty
Bose QuietComfort Ultra Earbuds,"features: CustomTune sound calibration, ANC, I...",silicone ear tips,1‑year limited warranty


In [152]:
# Pre-process the table
products_df = preprocess_table(products_df)
print("\n\nPre-Processed Table:")
products_df.head(5)



Pre-Processed Table:


,Name,attributes,material,warranty
0,Apple iPhone 15 Pro,"features: A17 Pro chip, 120 Hz ProMotion displ...","titanium frame, Ceramic Shield glass",1 year limited warranty
1,Samsung Galaxy S24 Ultra,"features: 200 MP camera, S Pen support, 120 Hz...","Armor Aluminum frame, Gorilla Glass Victus",1 year limited warranty
2,Google Pixel 8 Pro,"features: Tensor G3 chip, Magic Eraser, 50 MP ...","matte glass back, aluminum frame",1 year limited warranty
3,Sony WH 1000XM5 Headphones,"features: active noise cancelling, 30 hr batte...",synthetic leather earcups,1 year limited warranty
4,Bose QuietComfort Ultra Earbuds,"features: CustomTune sound calibration, ANC, I...",silicone ear tips,1 year limited warranty


In [153]:
# Extract the products' attributes.
# Notice that the attributes column in the table may contain multiple
# features, separated by ';', and each section's format is key:value.
# So each product is represented by a dictionary, of features' names
# and their corresponding values.
def break_attributes(attributes: str) -> dict:
    try:
        items = attributes.split(";")
        items = [item.split(":") for item in items]
        items = [item if len(item) > 1 else ("other", item[0]) for item in items]
        return {item[0].strip(): item[1].strip() for item in items}
    except Exception as e:
        raise Exception(f"Failed to process item {attributes}, e={e}")

def get_attributes_dict(r: pd.Series) -> dict:
    d = dict(r)
    d = {**d, **break_attributes(d["attributes"])}
    d.pop("attributes")
    return d

products_attributes = products_df.apply(lambda r: get_attributes_dict(r), axis=1).tolist()
print("Products Attributes:\n")
for i in range(3):
    print(f"\nProduct {i}:\n{json.dumps(products_attributes[i], indent=4)}")

Products Attributes:


Product 0:
{
    "Name": "Apple iPhone 15 Pro",
    "material": "titanium frame, Ceramic Shield glass",
    "warranty": "1 year limited warranty",
    "features": "A17 Pro chip, 120 Hz ProMotion display, USB C fast charging",
    "dimensions": "compact"
}

Product 1:
{
    "Name": "Samsung Galaxy S24 Ultra",
    "material": "Armor Aluminum frame, Gorilla Glass Victus",
    "warranty": "1 year limited warranty",
    "features": "200 MP camera, S Pen support, 120 Hz AMOLED",
    "other": "sustainably sourced"
}

Product 2:
{
    "Name": "Google Pixel 8 Pro",
    "material": "matte glass back, aluminum frame",
    "warranty": "1 year limited warranty",
    "features": "Tensor G3 chip, Magic Eraser, 50 MP camera",
    "rating": "4.7/5"
}


In [154]:
# Define the prompts
SYSTEM_PROMPT = """
You are an AI agent.
Your task is to describe products based on their given attributes.
The product to be described will be presented by a s JSON-formatted string, in
which the keys are the attributes names and the values are their description.

***For Example:***
"{
'Name':'Apple iPhone 15 Pro',
'material':'titanium...',
'warranty':'1‑year limited warranty',
'features':'A17 Pro chip...',
'battery':'long‐lasting',
}"

Notice that the first 4 attributes - Name, material, warranty, and features - appear,
for each product, while the other attributes appear alternately across the products.

Please generate for such product a description, which will satisfies the next criterions:

***Description Criterion:***
* Sounds natural and fluent, clear and easy to read.
* Should be grammatically correct, with puctuations and uppercase where needed.
* The 'tone' of the text should be friendly, sounds like a credible sales voice, nice and poilte.
* The length of the generated text, in words number, should be in the range 50-90 words.
* It is VERY IMPORTANT that the generated description can be inferred from the product attributes only, with no hallucinations!

***Output Format:***
Please retrive just the generated description.
"""

USER_PROMPT = """
Please describe the given product based on its attributes.
The attributes are provided below as JSON-Formatted string between triple backticks:

```json
"""

In [156]:
# Define the LLM client and the generation code.
import os
from openai import OpenAI

client = OpenAI(
    base_url="https://api.tokenfactory.nebius.com/v1/",
    api_key=os.environ.get("NEBIUS_API_KEY"),
)

def generate_response(attributes: dict,
                      model_name: str,
                      system_prompt: str,
                      user_prompt: str,
                      response_format: dict) -> dict:
    return client.chat.completions.create(
        model=model_name,
        response_format=response_format,
        messages=[
            {"role": "system", "content": system_prompt},
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": f"{user_prompt} {json.dumps(attributes)}```"}
                ],
            },
        ]
    ).to_dict()

In [158]:
# Define the response generation method.
import time

MODEL_NAME = "google/gemma-2-9b-it-fast"
IN_TOKEN_COST  = 0.00000003 # In token cost for Gemma-2-9b-it
OUT_TOKEN_COST = 0.00000009 # Out token cost for Gemma-2-9b-it
DESC_KEY = "generated_description"
LATENCY = "latency_ms"
NUM_IN_TOKENS = "input_tokens"
NUM_OUT_TOKENS = "output_tokens"
FLUENCY = "Fluency"
GRAMMAR = "Grammar"
TONE = "Tone"
LENGTH = "Length"
GROUNDING = "Grounding"
COST = "Cost"
DECISION = "final_score"

GEN_DESC_SCHEMA = {
  "name": "generated_description",
  "schema": {
    "type": "object",
    "properties": {
      "generated_description": {
        "type": "string",
        "description": "description of the given product based on its features",
        "minLength": 250,
        "maxLength": 750
      },
    },
    "required":["generated_description"],
    "additionalProperties": False
  },
  "strict": True
}

def generate_product_description(product: dict,
                                 model_name: str,
                                 system_prompt: str,
                                 user_prompt: str) -> dict:
    response_format = {"type": "json_schema", "json_schema": GEN_DESC_SCHEMA}
    start_time = time.time()
    response = generate_response(product, model_name, system_prompt, user_prompt, response_format=response_format)
    process_time = time.time() - start_time
    
    j = json.loads(response["choices"][0]["message"]["content"])
    return {
        DESC_KEY:j[GEN_DESC_FIELD],
        LATENCY:int(process_time * 1000),
        NUM_IN_TOKENS:response["usage"]["prompt_tokens"],
        NUM_OUT_TOKENS:response["usage"]["completion_tokens"],
        FLUENCY:"",
        GRAMMAR:"",
        TONE:"",
        LENGTH:"",
        GROUNDING:"",
        COST:"",
        DECISION:"",
    }

In [159]:
# Generate description for all the products.
descriptions = [generate_product_description(product, MODEL_NAME, SYSTEM_PROMPT, USER_PROMPT) for product in products_attributes]
descriptions_df = pd.DataFrame(descriptions)
descriptions_df.to_csv(DESCRIPTIONS_TABLE + "_task_2_repeat.csv")
print("The Descriptions Table - Phase I (Task 2):\n")
descriptions_df.head(5)

The Descriptions Table - Phase I (Task 2):



,generated_description,latency_ms,input_tokens,output_tokens,Fluency,Grammar,Tone,Length,Grounding,Cost,final_score
0,"Introducing the Apple iPhone 15 Pro, a powerfu...",1658,675,91,,,,,,,
1,The Samsung Galaxy S24 Ultra boasts a sleek de...,1638,676,85,,,,,,,
2,The Google Pixel 8 Pro boasts a sleek design w...,1433,672,104,,,,,,,
3,Enjoy your music in unparalleled comfort with ...,1550,673,120,,,,,,,
4,Introducing the Bose QuietComfort Ultra Earbud...,1418,664,118,,,,,,,


<h2 style="text-align: center;">
  <span style="display: inline-block; border-bottom: 2px solid black; padding-bottom: 5px;">
    Task 3
  </span>
</h2>

Manual analysis of the first 14 products in the table.

The descriptions induced by the initial configuration seemed very good, and 
all but one got Pass score.

The analysis itself was rather tedious, and the majority of efforts was dedicated for verifying that no
hallusinations were inserted to the generated texts.

In [160]:
#Present the manually analyzed products:
manual_analyzed_df = pd.read_csv(DESCRIPTIONS_TABLE + "_human_eval_task_3.csv", index_col=0)
manual_analyzed_df[:14].head(15)

,generated_description,latency_ms,input_tokens,output_tokens,Fluency,Grammar,Tone,Length,Grounding,Latency,Cost,final_score
0,The Apple iPhone 15 Pro is a sleek and powerfu...,1184,675,87,Good,Good,OK,Good,Good,OK,Good,Pass
1,The Samsung Galaxy S24 Ultra is crafted with a...,1121,676,101,Good,Good,OK,Good,Good,OK,Good,Pass
2,The Google Pixel 8 Pro is a sleek and stylish ...,955,672,105,Good,Good,Good,Good,Good,Good,Good,Pass
3,Experience superior comfort and immersive soun...,1277,673,104,Good,Good,Good,Good,Good,OK,Good,Pass
4,Experience exceptional audio clarity and immer...,1333,664,111,Good,Good,Good,Good,Good,OK,Good,Pass
5,"Meet the Amazon Echo Dot (5th Gen), a smart sp...",1447,665,128,Good,Good,Good,OK,Good,OK,OK,Pass
6,The Dell XPS 13 9310 Laptop is a sleek and por...,1125,692,102,Good,Good,Good,Good,Good,OK,Good,Pass
7,"The Apple MacBook Air 13"" (M3) is crafted from...",1021,688,112,Good,Good,Good,Good,Good,OK,OK,Pass
8,The Microsoft Surface Pro 10 is a sleek and po...,993,678,88,Good,Good,Good,Good,Good,Good,Good,Pass
9,The Garmin Forerunner 255 is a reliable runnin...,791,671,87,Good,Good,Good,Good,Good,Good,Good,Pass


<h2 style="text-align: center;">
  <span style="display: inline-block; border-bottom: 2px solid black; padding-bottom: 5px;">
    Task 4 - Improvement Cycle
  </span>
</h2>

As the descritions generated in the first iteration (Task 2) seemed to me satisfactory,
I did not see much point in further tuning and analysis of the generation process itself
(i.e. prompts engineering, hyper parameters tuning, etc.).

What made me more curious, though, was the impact of larger models on the latency and cost
criterions.
So I tested larger models of Gemma and GPT.

Comparing the generated texts of these larger model to the texts generated by the base model
(Gemma-2-9b-it) was challenging: the texts looked very similar.
Indeed, the style of the texts generated by the larger models were evidently a bit more flashy
and bombastic, but it is not necessarily better, and depends on the specific requirements and
circumstances of the business.

However, the impact of the larger models on the the latency and cost criterions was drammatic
and harmful: all descriptions failed due to high cost.
In the case of the larger Gemma model, there were also many failures due to high latency.

As the criterions for latency and cost were set based on the performance of the initial model,
it is pretty trivial that larger models failed on these criterions.

However, also here it is a matter of specific business requirements and circumstance - the criterions
were set based on the best available judgement, and could of course be different, given other
instructions or details.

In [31]:
MODEL_NAME = "openai/gpt-oss-120b-fast"
IN_TOKEN_COST  = 0.0000001 # In token cost for gpt-oss-120b-fast
OUT_TOKEN_COST = 0.0000005 # Out token cost for gpt-oss-120b-fast

descriptions = [generate_product_description(product, MODEL_NAME, SYSTEM_PROMPT, USER_PROMPT) for product in products_attributes]
descriptions_df = pd.DataFrame(descriptions)
print("The Descriptions Table - Phase I (Task 2):\n")
descriptions_df.head(5)

The Descriptions Table - Phase I (Task 2):



,generated_description,latency_ms,input_tokens,output_tokens,Fluency,Grammar,Tone,Length,Grounding,Cost,final_score
0,"The Apple iPhone 15 Pro combines a sleek, comp...",2009,414,122,,,,,,,
1,The Samsung Galaxy S24 Ultra combines a sleek ...,1124,412,139,,,,,,,
2,"The Google Pixel 8 Pro combines sleek, matte‑g...",1039,413,148,,,,,,,
3,"The Sony WH‑1000XM5 headphones combine sleek, ...",1315,412,160,,,,,,,
4,The Bose QuietComfort Ultra Earbuds combine sl...,1024,407,139,,,,,,,


In [32]:
descriptions_df.to_csv(DESCRIPTIONS_TABLE + "_task_4_gpt_0ss_120b.csv")

<h2 style="text-align: center;">
  <span style="display: inline-block; border-bottom: 2px solid black; padding-bottom: 5px;">
    Task 5 - Judge Agent
  </span>
</h2>



In [161]:
# Generate the product entries, which should be sent to the judge model.
descriptions_df = pd.read_csv(DESCRIPTIONS_TABLE + "_human_eval_task_3.csv", index_col=0)
descriptions_df.head(14)

,generated_description,latency_ms,input_tokens,output_tokens,Fluency,Grammar,Tone,Length,Grounding,Latency,Cost,final_score
0,The Apple iPhone 15 Pro is a sleek and powerfu...,1184,675,87,Good,Good,OK,Good,Good,OK,Good,Pass
1,The Samsung Galaxy S24 Ultra is crafted with a...,1121,676,101,Good,Good,OK,Good,Good,OK,Good,Pass
2,The Google Pixel 8 Pro is a sleek and stylish ...,955,672,105,Good,Good,Good,Good,Good,Good,Good,Pass
3,Experience superior comfort and immersive soun...,1277,673,104,Good,Good,Good,Good,Good,OK,Good,Pass
4,Experience exceptional audio clarity and immer...,1333,664,111,Good,Good,Good,Good,Good,OK,Good,Pass
5,"Meet the Amazon Echo Dot (5th Gen), a smart sp...",1447,665,128,Good,Good,Good,OK,Good,OK,OK,Pass
6,The Dell XPS 13 9310 Laptop is a sleek and por...,1125,692,102,Good,Good,Good,Good,Good,OK,Good,Pass
7,"The Apple MacBook Air 13"" (M3) is crafted from...",1021,688,112,Good,Good,Good,Good,Good,OK,OK,Pass
8,The Microsoft Surface Pro 10 is a sleek and po...,993,678,88,Good,Good,Good,Good,Good,Good,Good,Pass
9,The Garmin Forerunner 255 is a reliable runnin...,791,671,87,Good,Good,Good,Good,Good,Good,Good,Pass


In [162]:
# Extract from the table details that are needed for rating some criterions.
IN_TOKEN_COST  = 0.00000003 # In token cost for Gemma-2-9b-it
OUT_TOKEN_COST = 0.00000009 # Out token cost for Gemma-2-9b-it
generated_descriptions = descriptions_df.generated_description.tolist()
latencies = [int(x) for x in descriptions_df.latency_ms.tolist()]
inp_tokens = [int(x) for x in descriptions_df.input_tokens.tolist()]
out_tokens = [int(x) for x in descriptions_df.output_tokens.tolist()]
costs = [x*IN_TOKEN_COST + y*OUT_TOKEN_COST for x,y in zip(inp_tokens, out_tokens)]
print("Generated Descriptions:\n")
print(generated_descriptions[0])
print(generated_descriptions[1])
print(generated_descriptions[2])
print(f"\nLatency:\n{latencies[:3]},\n\nInput Tokens:\n{inp_tokens[:3]},\n\nOut Tokens:\n{out_tokens[:3]}\n")

Generated Descriptions:

The Apple iPhone 15 Pro is a sleek and powerful device featuring a titanium frame and Ceramic Shield glass for ultimate durability.  This compact powerhouse boasts the impressive A17 Pro chip, a 120 Hz ProMotion display for smooth scrolling, and  USB C fast charging for convenience.  Your iPhone 15 Pro comes with a 1-year limited warranty, giving you peace of mind. 



The Samsung Galaxy S24 Ultra is crafted with a premium Armor Aluminum frame and scratch-resistant Gorilla Glass Victus for a robust and elegant look. It's backed by a 1 year limited warranty. Capture stunning photos with its groundbreaking 200MP camera, and enjoy smooth, responsive navigation with the integrated S Pen and incredible 120Hz AMOLED display. Plus, you can feel good knowing your Galaxy S24 Ultra is responsibly made using sustainably sourced materials.  



The Google Pixel 8 Pro is a sleek and stylish smartphone featuring a matte glass back and an aluminum frame.  It boasts powerful p

In [165]:
# The prompts for the judge model.
JUDGE_SYSTEM_PROMPT = """
You are an AI agent who provides a judging service.
Your task is to rate a given text, which was generated
by another AI agent.
The generated text is a description of an e-commerce product, which
is based on a set of attributes characterizing the product.
The product to be described will be presented by a s JSON-formatted string,
consisting on two keys: attribute and generated_description.
Here is an example of the product input:

*** Example:***
"{
'attributes':
    {
        'Name':'Apple iPhone 15 Pro',
        'material':'titanium...',
        'warranty':'1-year limited warranty',
        'features':'A17 Pro chip...',
        'battery':'long-lasting',
    }
'generated_description': 'The Apple iPhone 15 Pro is a sleek and powerful device featuring...'
}"

Notice that the first 4 attributes - Name, material, warranty, and features - appear,
for each product, while the other attributes appear alternately across the products.

Your task, therefore, is to rate the generated_description based on the product's attributes.
You should assign to the generated description four scores, according to the next
scoring rubric and accoring to the provided response_format structure.

***Generated Description Scoring rubric:***
* Fluency:
    * Good: if the generated description is coherent, sounds natural, clear, and easy to read.
    * OK: if the generated description is still as above, but slighly less fluent and clear.
    * Bad: if there are gliches of uncoherent text, or awkward language.
* Grammar:
    * Good: if the generated description is grammatically correct, with punctuations, and uppercase where needed.
    * OK: if the generated description is still basically correct, but with minor glich or two (e.g. none perfect punctuation).
    * Bad: if there are more than two grammaticall gliches in the generated description.
* Tone:
    * Good: if the generated description matches friendly, credible sales voice, persuasive, and include poilte words and subtle enthusiasm.
    * OK: if the generated description is still nice and decent, but lacking some of the aspects mentioned in 'Good' above.
    * Bad: if the generated description lacks too many positive aspects, described above, or contains some less friendly expressions.
* Grounding:
    * Good: if there are no hallucinations, and the generated description can be directly inferred from the product name or features.
    * OK: if the above conditions are satisfied, but with possible minor 'creativity' (e.g. use synonyms, 'metal' instead of 'aluminum').
    * Bad: if the generated description contains hallucinations, or it can be hardly inferred from the content.

***Output Format:***
Please retrive a response according to the provided response_format structure.
Please provide for each criterion an explanation, which explains your verdict choice.
"""

JUDGE_USER_PROMPT = """
Please score the given product's generated description based on its attributes,
as explained above, and output your decision according to the provided structure.
The attributes are provided below as JSON-Formatted string between triple backticks:

```json
"""

In [166]:
# Prepare the input for the judge model
judge_input_entries = [{ATTR_FIELD:x, GEN_DESC_FIELD:y} for x,y in zip(products_attributes, generated_descriptions)]
for i in range(3):
    print(f"\nProduct {i}:\n{json.dumps(judge_input_entries[i], indent=4)}")


Product 0:
{
    "attributes": {
        "Name": "Apple iPhone 15 Pro",
        "material": "titanium frame, Ceramic Shield glass",
        "warranty": "1 year limited warranty",
        "features": "A17 Pro chip, 120 Hz ProMotion display, USB C fast charging",
        "dimensions": "compact"
    },
    "generated_description": "The Apple iPhone 15 Pro is a sleek and powerful device featuring a titanium frame and Ceramic Shield glass for ultimate durability.  This compact powerhouse boasts the impressive A17 Pro chip, a 120 Hz ProMotion display for smooth scrolling, and  USB C fast charging for convenience.  Your iPhone 15 Pro comes with a 1-year limited warranty, giving you peace of mind. \n\n\n"
}

Product 1:
{
    "attributes": {
        "Name": "Samsung Galaxy S24 Ultra",
        "material": "Armor Aluminum frame, Gorilla Glass Victus",
        "warranty": "1 year limited warranty",
        "features": "200 MP camera, S Pen support, 120 Hz AMOLED",
        "other": "sustainably so

In [168]:
# Define the response schema for the jusdge model
JUDGE_RESPONSE_SCHEMA = {
  "name": "generated_description_judgement",
  "schema": {
    "type": "object",
    "properties": {
      "fluency": {
        "type": "object",
        "description": "rate the fluency level of the generated text",
        "properties": {
          "justification": {
            "type": "string",
            "description": "explain why you chose this fluency rate for the text",
            "minLength": 75
          },
          "verdict": {
            "type": "string",
            "description": "the fluency rate of the text: Good / OK / Bad"
          },
        },
        "required": [
          "justification",
          "verdict"
        ],
        "additionalProperties": False
      },
      "grammar": {
        "type": "object",
        "description": "rate the level of the grammar accuracy of the generated text",
        "properties": {
          "justification": {
            "type": "string",
            "description": "explain why you chose this grammar accuracy rate for the text",
            "minLength": 75
          },
          "verdict": {
            "type": "string",
            "description": "the grammar accuracy rate of the text: Good / OK / Bad"
          },
        },
        "required": [
          "justification",
          "verdict"
        ],
        "additionalProperties": False
      },
      "tone": {
        "type": "object",
        "description": "rate the tone quality of the generated text",
        "properties": {
          "justification": {
            "type": "string",
            "description": "explain why you chose this rate for the tone quality of the text",
            "minLength": 75
          },
          "verdict": {
            "type": "string",
            "description": "the rate of the tone quality of the text: Good / OK / Bad"
          },
        },
        "required": [
          "justification",
          "verdict"
        ],
        "additionalProperties": False
      },
      "grounding": {
        "type": "object",
        "description": "rate the faithfulness of the generated text",
        "properties": {
          "justification": {
            "type": "string",
            "description": "explain why you chose this rate for the faithfulness of the text",
            "minLength": 75
          },
          "verdict": {
            "type": "string",
            "description": "the rate of the faithfulness of the text: Good / OK / Bad"
          },
        },
        "required": [
          "justification",
          "verdict"
        ],
        "additionalProperties": False
      }
   },
    "required": [
      "fluency",
      "grammar",
      "tone",
      "grounding"
    ],
    "additionalProperties": False
  },
  "strict": True
}

In [170]:
# Judge the generated descriptions.
def judge_product_description(product: dict, model_name: str, system_prompt: str, user_prompt: str) -> dict:
    response_format = {"type": "json_schema", "json_schema": JUDGE_RESPONSE_SCHEMA}
    response = generate_response(product, model_name, system_prompt, user_prompt, response_format=response_format)
    return response

response = judge_product_description(judge_input_entries[0], model_name="meta-llama/Meta-Llama-3.1-8B-Instruct",system_prompt=JUDGE_SYSTEM_PROMPT, user_prompt=JUDGE_USER_PROMPT)
print(response["choices"][0]["message"]["content"])

{
  "fluency": {
    "justification": "The generated description reads naturally, and it's easy to follow the text. However, the description is slightly less fluent because the last sentence feels a bit disconnected from the rest. The transition to the warranty information could be smoother. Despite this, the overall coherence and flow of the text make it 'OK'.",
    "verdict": "OK"
  },
  "grammar": {
    "justification": "The generated description only has minor grammatical errors, such as the use of a period instead of a comma, and a slight inaccuracy in phrase order. However, the presence of serial commas can also be seen as minor lint rather than an error of the actual grammar's structural level. Thus, I would still grant this text 'OK' on this fairness",
    "verdict": "OK"
  },
  "tone": {
    "justification": "The generated description uses a friendly tone that is pleasant to read. It emphasizes the positive aspects of the product, like its durability and convenience features, 